# J1939 — trucks, buses and machines

A diagnostic tool and a simulated heavy-duty engine ECU share a virtual 250 kbit/s CAN bus. Replace the bus with `await CanBus.OpenAsync("socketcan:can0")` (or `slcan:`, `gsusb:`, `pcan:usb1`) to listen to a real vehicle — only vehicles you are authorised to work on.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.17.0-preview.1"

## The 29-bit identifier
Priority, parameter group number (PGN) and source address; PDU1 PGNs (PF < 240) also carry a destination.

In [ ]:
using IoTCom.Net.Protocols.J1939;
using IoTCom.Net.Transport.Can;

var eec1Id = J1939Id.FromCanId(0x0CF00400);
Console.WriteLine($"priority {eec1Id.Priority}, PGN {eec1Id.Pgn} ({Pgn.Name(eec1Id.Pgn)}), source 0x{eec1Id.Source:X2}");
var reqId = J1939Id.FromCanId(0x18EA00F9);
Console.WriteLine($"PGN {reqId.Pgn} ({Pgn.Name(reqId.Pgn)}) from 0x{reqId.Source:X2} to 0x{reqId.Destination:X2}");

## SPNs: scaling, offsets and 'not available'
Each parameter has a fixed position, resolution and offset. 0xFF… means not available.

In [ ]:
foreach (var v in J1939Spn.Decode(Pgn.Eec1, Convert.FromHexString("F17D82A02200FF7D")))
    Console.WriteLine(v);
var et1 = new byte[8];
et1.AsSpan().Fill(0xFF);
J1939Spn.Encode(Pgn.Et1, 110, 88, et1);
Console.WriteLine($"ET1 with coolant 88 °C: {Convert.ToHexString(et1)}");

## A node, an engine and a VIN over RTS/CTS
The tool claims address 0xF9; the VIN is longer than 8 bytes, so the engine answers with the transport protocol.

In [ ]:
var jNet = new VirtualCanNetwork("notebook-j1939");
var truck = J1939EngineSimulator.Create(jNet.CreateNode());
truck.Throttle = 50;
var tool = J1939Node.Create(jNet.CreateNode(), o => o.ReadOnly = true);
await tool.StartAsync();
await truck.StartAsync();
var vinMsg = await tool.RequestAsync(Pgn.VehicleIdentification, 0x00);
Console.WriteLine($"VIN: {System.Text.Encoding.ASCII.GetString(vinMsg.Data)} ({vinMsg.Data.Length} bytes)");
foreach (var (addr, name) in tool.Claims) Console.WriteLine($"0x{addr:X2}: function {name.Function}, identity {name.IdentityNumber}");

## Broadcasts and DM1
The engine broadcasts EEC1/CCVS1 every 100 ms and DM1 every second. Start an oil leak and watch the amber lamp.

In [ ]:
J1939Dm1? lastDm1 = null;
double rpmNow = 0;
tool.MessageReceived += m =>
{
    if (m.Pgn == Pgn.Dm1) lastDm1 = J1939Dm1.Parse(m.Data);
    if (m.Pgn == Pgn.Eec1) rpmNow = m.Values.First(v => v.Spn == 190).Value ?? 0;
};
truck.OilLeak();
for (var i = 0; i < 40; i++) truck.Step(1);
await Task.Delay(1300);
Console.WriteLine($"{rpmNow:0} rpm, amber lamp {lastDm1?.AmberWarningLamp}");
foreach (var d in lastDm1!.Dtcs) Console.WriteLine($"SPN {d.Spn} {J1939Spn.Name(d.Spn)}: FMI {d.Fmi} {d.FailureMode}");

## On the wire

In [ ]:
foreach (var f in J1939Spn.Describe(new CanFrame(0x18FECA00, Convert.FromHexString("04FF640001010000"), CanFrameFlags.Extended)))
    Console.WriteLine($"{f.Name,-10} {f.Value}");
await tool.DisposeAsync();
await truck.DisposeAsync();

## Going further
`iotcom j1939 monitor --can sim`, `iotcom j1939 request vin --to 00 --can sim`, `iotcom j1939 claims --can sim`, the Gallery's *Truck cluster over J1939* and the J1939Monitor sample. See `docs/en/protocols/j1939.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*